In [ ]:
%load_ext autoreload
%autoreload 2

from radar.components import geometry
from radar.utils.calculate import convert
from radar.utils.typing.enums import FrequencyUnit
from radar.utils.typing.units import Frequency

from radar.components import Element
from radar.utils.calculate import pattern
from radar.utils.typing import (
    PhaseUnit,
    DirectionDomain,
    FigureType,
    AmplitudeDomain,
    Angle,
    AmplitudeUnit,
)

from radar.components.array import Array
from radar.components.response import FrequencyResponse

import polars as pl

from radar.utils.typing.constants import DataHeader

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Create Element

And show its beam pattern

In [68]:
beam_width = 5
beam_width_tuple = (
    Angle(beam_width, PhaseUnit.DEGREE),
    Angle(beam_width, PhaseUnit.DEGREE),
)

az_bound = 90
el_bound = 90
az_bound_tuple = (Angle(-az_bound, PhaseUnit.DEGREE), Angle(az_bound, PhaseUnit.DEGREE))
el_bound_tuple = (Angle(-el_bound, PhaseUnit.DEGREE), Angle(el_bound, PhaseUnit.DEGREE))


element_pattern = pattern.Isotropic()
freq = Frequency(1, FrequencyUnit.GIGAHERTZ)

antenna_element = Element(element_pattern, az_bound_tuple, el_bound_tuple, freq, 1)

antenna_element.plot.beam(
    DirectionDomain.ANGLE,
    PhaseUnit.DEGREE,
    AmplitudeDomain.Gain,
    AmplitudeUnit.DECIBEL,
    FigureType.IMAGE,
    freq,
)

## Define Array Structure

In [69]:
cf = Frequency(1, FrequencyUnit.GIGAHERTZ)
distance = convert.cf_to_min_dist(cf)
geometry = geometry.Grid(10, 10, distance)

## Define Array

using the specificed geometry and element

In [70]:
arr = Array(antenna_element, geometry)

### Investigate Array Geometry

In [71]:
arr.plot.geometry()

### Investigate Array Beam Pattern

#### Image

In [72]:
arr.plot.beam(
    DirectionDomain.ANGLE,
    PhaseUnit.DEGREE,
    AmplitudeDomain.Gain,
    AmplitudeUnit.DECIBEL,
    FigureType.IMAGE,
    freq,
)

#### Surface

In [73]:
arr.plot.beam(
    DirectionDomain.ANGLE,
    PhaseUnit.DEGREE,
    AmplitudeDomain.Gain,
    AmplitudeUnit.DECIBEL,
    FigureType.SURFACE,
    freq,
    (Angle(20, PhaseUnit.DEGREE), Angle(20, PhaseUnit.DEGREE)),
)

## Array Factor as a Function of Frequency

In [74]:
f1 = Frequency(0.1, FrequencyUnit.MEGAHERTZ)
f2 = Frequency(5, FrequencyUnit.GIGAHERTZ)
df = pl.DataFrame(
    {DataHeader.FREQ_GAIN_DB: [0, 0], DataHeader.FREQ_FREQS: [f1.Hz, f2.Hz]}
)

f_resp = FrequencyResponse(df=df)

antenna_element = Element(element_pattern, az_bound_tuple, el_bound_tuple, f_resp, 1)
arr = Array(antenna_element, geometry)

#### Array Factor Below Operational Frequency

In [75]:
arr.plot.beam(
    DirectionDomain.ANGLE,
    PhaseUnit.DEGREE,
    AmplitudeDomain.AntennaFactor,
    AmplitudeUnit.DECIBEL,
    FigureType.SURFACE,
    Frequency(0.1, FrequencyUnit.GIGAHERTZ),
)

#### Array Factor At Design Frequency

In [76]:
arr.plot.beam(
    DirectionDomain.ANGLE,
    PhaseUnit.DEGREE,
    AmplitudeDomain.AntennaFactor,
    AmplitudeUnit.DECIBEL,
    FigureType.SURFACE,
    Frequency(1, FrequencyUnit.GIGAHERTZ),
)

#### Array Factor At Above Operational Frequency

In [77]:
arr.plot.beam(
    DirectionDomain.ANGLE,
    PhaseUnit.DEGREE,
    AmplitudeDomain.AntennaFactor,
    AmplitudeUnit.DECIBEL,
    FigureType.SURFACE,
    Frequency(5, FrequencyUnit.GIGAHERTZ),
)